# Adversarial robustness & certification

Certification proves a prediction cannot change inside a specified perturbation radius.

The notebook compares empirical FGSM robustness with a conservative margin certificate. A certificate is only a proof for the named norm, radius, and bound. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 19
rng = np.random.default_rng(SEED)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def make_group(X, y):
    scores = X[:, 0]
    if len(np.unique(scores)) < 3:
        scores = X.sum(axis=1)
    cutoff = np.median(scores)
    group = (scores > cutoff).astype(int)
    if len(np.unique(group)) < 2:
        group = (np.arange(len(y)) % 2).astype(int)
    return group


def safe_split(X, y, group=None, test_size=0.4):
    stratify = y
    if min(np.bincount(y.astype(int))) < 2:
        stratify = None
    pieces = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=test_size,
        random_state=0,
        stratify=stratify,
    )
    x_train, x_test, y_train, y_test, idx_train, idx_test = pieces
    if group is None:
        return x_train, x_test, y_train, y_test, idx_train, idx_test, None, None
    return x_train, x_test, y_train, y_test, idx_train, idx_test, group[idx_train], group[idx_test]


def fit_scaled_logreg(X, y, C=1.0):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, C=C, multi_class="auto")
    model.fit(x_train_s, y_train)
    return model, scaler, x_train_s, x_test_s, y_train, y_test, idx_train, idx_test


def probability_for_label(model, X, y):
    probs = model.predict_proba(X)
    positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    return probs[np.arange(len(y)), positions]


def fgsm_attack(model, X, y, eps):
    probs = model.predict_proba(X)
    class_positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), class_positions] = 1.0
    grad = (probs - one_hot) @ model.coef_
    direction = np.sign(grad)
    return X + eps * direction


def robust_accuracy_for_eps(model, X, y, eps):
    attacked = fgsm_attack(model, X, y, eps)
    preds = model.predict(attacked)
    return accuracy_score(y, preds)


def fairness_report(y, yhat, group):
    rows = {}
    positive = int(np.max(y))
    for g in [0, 1]:
        mask = group == g
        truth_pos = mask & (y == positive)
        truth_neg = mask & (y != positive)
        pred_pos = mask & (yhat == positive)
        tp = int(np.sum(pred_pos & truth_pos))
        fp = int(np.sum(pred_pos & truth_neg))
        fn = int(np.sum((~pred_pos) & truth_pos))
        tn = int(np.sum((~pred_pos) & truth_neg))
        rate = float(np.mean(yhat[mask] == positive)) if np.any(mask) else np.nan
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        rows[g] = {"n": int(mask.sum()), "pos_rate": rate, "tpr": tpr, "fpr": fpr, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
    dp_gap = abs(rows[0]["pos_rate"] - rows[1]["pos_rate"])
    eo_gap = max(abs(rows[0]["tpr"] - rows[1]["tpr"]), abs(rows[0]["fpr"] - rows[1]["fpr"]))
    return {"group0": rows[0], "group1": rows[1], "dp_gap": dp_gap, "eo_gap": eo_gap}


def plot_2d_projection(ax, X, color, title):
    if X.shape[1] >= 2:
        shown = X[:, :2]
    else:
        shown = np.column_stack([X[:, 0], np.zeros(len(X))])
    ax.scatter(shown[:, 0], shown[:, 1], c=color, s=18, cmap="viridis", alpha=0.75)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once: margin certificate

The certificate checks $$f_y(x)-\max_{k\ne y}f_k(x) \gt 2L\varepsilon.$$ It is conservative: if the inequality fails, the example may still be robust, but this proof did not certify it.

In [ ]:

components = np.array([1.2, 0.7, 0.3], dtype=float)
knob = 0.2
total = float(np.sum(components))
absolute_total = float(np.sum(np.abs(components)))
leading_share = float(abs(components[0]) / absolute_total)
guarded = float(total + knob * absolute_total)
contrast = float(total - components[2])
change = float(contrast - total)

assert np.isclose(total, 2.2)
assert np.isclose(absolute_total, 2.2)
assert np.isclose(round(guarded, 3), 2.64)
assert np.isclose(round(leading_share, 3), 0.545)
assert np.isclose(total, 2.200)

print("total", round(total, 3))
print("absolute_total", round(absolute_total, 3))
print("leading_share", round(leading_share, 3))
print("guarded", round(guarded, 3))
print("change_without_component_3", round(change, 3))


For a linear softmax classifier in standardized space, a conservative local score-movement bound uses the largest pairwise coefficient distance as $L$.

In [ ]:

def certify_margin(scores, y, L, eps, classes):
    class_positions = np.array([np.where(classes == label)[0][0] for label in y])
    true_scores = scores[np.arange(len(y)), class_positions]
    masked = scores.copy()
    masked[np.arange(len(y)), class_positions] = -np.inf
    runner_up = np.max(masked, axis=1)
    margins = true_scores - runner_up
    certified = margins > 2.0 * L * eps
    return certified, margins

X, y = clf_ladder()[1][1:]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
L = float(np.max(np.linalg.norm(model.coef_[:, None, :] - model.coef_[None, :, :], ord=2, axis=2)))
certified, margins = certify_margin(model.decision_function(x_test), y_test, L, 0.05, model.classes_)
print("L_bound", round(L, 3))
print("certified_fraction", round(float(np.mean(certified)), 3))
print("median_margin", round(float(np.median(margins)), 3))


## The dataset ladder

The same classifier family is tested on D1-D5: a hand toy, synthetic blobs, noisy moons, real Wine data, and real Breast Cancer data.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    print(name)
    print("  shape", X.shape)
    print("  classes", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same certification across D1-D5

The metric is certified accuracy: the fraction of all test examples that are both correctly predicted and certified at the declared epsilon.

In [ ]:

epsilon = 0.05
cert_results = []
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    scores = model.decision_function(x_test)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    if len(model.classes_) == 2 and model.coef_.shape[0] == 1:
        coef = np.vstack([-model.coef_[0], model.coef_[0]])
    else:
        coef = model.coef_
    L = float(np.max(np.linalg.norm(coef[:, None, :] - coef[None, :, :], ord=2, axis=2)))
    certified, margins = certify_margin(scores, y_test, L, epsilon, model.classes_)
    correct = model.predict(x_test) == y_test
    empirical = robust_accuracy_for_eps(model, x_test, y_test, epsilon)
    cert_acc = float(np.mean(certified & correct))
    cert_results.append({"rung": rung, "name": name, "empirical": empirical, "certified": cert_acc, "L": L})
print("rung | empirical_robust | certified_accuracy | L")
for row in cert_results:
    print(row["rung"], round(row["empirical"], 3), round(row["certified"], 3), round(row["L"], 3))


## Results visualization

Left: empirical robust versus certified bars. Right: certified accuracy falls as epsilon grows.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, cert_results):
    ax.bar(["empirical", "certified"], [row["empirical"], row["certified"]], color=["tab:blue", "tab:purple"])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"D{row['rung']}")
plt.tight_layout()
plt.show()

eps_grid = np.linspace(0.0, 0.20, 6)
fig, ax = plt.subplots(figsize=(7, 4))
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    scores = model.decision_function(x_test)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    if len(model.classes_) == 2 and model.coef_.shape[0] == 1:
        coef = np.vstack([-model.coef_[0], model.coef_[0]])
    else:
        coef = model.coef_
    L = float(np.max(np.linalg.norm(coef[:, None, :] - coef[None, :, :], ord=2, axis=2)))
    curve = []
    correct = model.predict(x_test) == y_test
    for eps in eps_grid:
        certified, margins = certify_margin(scores, y_test, L, eps, model.classes_)
        curve.append(float(np.mean(certified & correct)))
    ax.plot(eps_grid, curve, marker="o", label=f"D{rung}")
ax.set_xlabel("epsilon")
ax.set_ylabel("certified accuracy")
ax.legend()
plt.show()


## Pitfall on D5: claiming certification outside the norm ball

The wrong statement treats a certified point at one radius as certified everywhere. The fix labels the norm, radius, and conservative $L$ for each certificate.

In [ ]:

name, X, y = clf_ladder()[-1]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
scores = model.decision_function(x_test)
scores = np.column_stack([-scores, scores]) if scores.ndim == 1 else scores
coef = np.vstack([-model.coef_[0], model.coef_[0]]) if model.coef_.shape[0] == 1 else model.coef_
L = float(np.max(np.linalg.norm(coef[:, None, :] - coef[None, :, :], ord=2, axis=2)))
wrong_eps = 0.05
claimed_eps = 0.25
certified_small, margins = certify_margin(scores, y_test, L, wrong_eps, model.classes_)
certified_large, margins = certify_margin(scores, y_test, L, claimed_eps, model.classes_)
print("wrong_certified_at_0_05_then_claimed_0_25", round(float(np.mean(certified_small)), 3))
print("fixed_certified_linf_radius_0_25", round(float(np.mean(certified_large)), 3))
print("bound_L", round(L, 3), "norm", "standardized_l2_score_bound")


## Evaluate it + Practice

- Metric: certified accuracy at a named epsilon.
- No-skill baseline: empirical robust accuracy is not a proof.
- Cheap sanity check: epsilon zero certificate should track positive margins on correct predictions.
- Ablation: use an artificially large L and watch certificates vanish.
- Failure signals: certificates are averaged without exposing uncertified high-risk examples.

Practice prompts:
1. Change one stress knob and predict the direction of the metric before running.


2. Compare the empirical FGSM curve with the certificate curve on D4.

3. Print the smallest certified margin and inspect why it fails first.